# RAG Workshop: from PDF to answer

**Run it:** click each cell and press ▶ (or Shift+Enter), top to bottom. No setup needed.

```
PDFs → load → chunk → embed → vector store ──┐
                                              ├→ retrieve top-k → prompt (context + question) → LLM → answer
question → embed ─────────────────────────────┘
```

## 0. Install (about 1 to 2 minutes the first time)

> Red lines starting with `ERROR: pip's dependency resolver...` are expected in Colab. They only mention Colab's own tools, so ignore them and continue. If a later cell shows an import error, choose **Runtime → Restart session** and run again from step 1 (skip this install cell).

In [ ]:
!pip install -q langchain-community langchain-text-splitters pypdf chromadb sentence-transformers langchain-openai pandas

## 1. Data ingestion: read documents into Python

A `Document` is just text plus **metadata** (where it came from, which page).
Upload your own PDFs or .txt files. If you skip the upload, a small Boston example is used.

In [ ]:
import os, glob
from pathlib import Path
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader, TextLoader

DATA_DIR = Path("my_files"); DATA_DIR.mkdir(exist_ok=True)

try:                                   # inside Colab: open the upload dialog
    from google.colab import files
    for name, content in files.upload().items():
        (DATA_DIR / name).write_bytes(content)
except ImportError:                    # running locally
    pass

docs = []
for path in sorted(DATA_DIR.glob("**/*")):
    if path.suffix.lower() == ".pdf":
        pages = PyPDFLoader(str(path)).load()
    elif path.suffix.lower() in {".txt", ".md"}:
        pages = TextLoader(str(path), encoding="utf-8").load()
    else:
        continue
    for p in pages:
        p.metadata["source_file"] = path.name
    docs.extend(pages)

if not docs:   # nothing uploaded: use a built-in example
    sample = [
        "The Freedom Trail is a 2.5 mile red brick line through Boston connecting 16 historic sites from the American Revolution.",
        "The USS Constitution is a famous ship nicknamed Old Ironsides because it has never been defeated in battle.",
        "The Museum of Science has interactive exhibits, a butterfly garden, 4-D films, IMAX films and planetarium shows.",
        "The Isabella Stewart Gardner Museum was the site of the biggest unsolved art theft in history, in 1990.",
        "Faneuil Hall has stood since 1742 and is home to shops and restaurants. Quincy Market is right behind it.",
        "The New England Aquarium is a research and conservation group with a Giant Ocean Tank.",
    ]
    docs = [Document(page_content=t, metadata={"source_file": "boston_example", "page": i}) for i, t in enumerate(sample)]

print(f"Loaded {len(docs)} pages/documents")
docs[0]

## 2. Chunking: cut the text into pieces

Long texts do not fit in one search, so we cut them into **chunks**.
`chunk_overlap` repeats a few characters at the start of the next chunk so sentences are not cut in half.

Try changing `CHUNK_SIZE` and `CHUNK_OVERLAP` and re-run this cell and the ones below.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import pandas as pd

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", " ", ""],   # try paragraphs first, then lines, then words
)
chunks = splitter.split_documents(docs)
print(f"{len(docs)} documents -> {len(chunks)} chunks")

pd.set_option("display.max_colwidth", 110)
pd.DataFrame({
    "file": [c.metadata["source_file"] for c in chunks],
    "page": [c.metadata.get("page") for c in chunks],
    "chars": [len(c.page_content) for c in chunks],
    "text": [c.page_content for c in chunks],
}).head(10)

## 3. Embeddings: turn text into numbers

An **embedding** is a list of numbers (here 384) that captures meaning.
Texts with similar meaning get similar numbers, even with different words.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [c.page_content for c in chunks]
embeddings = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
print("shape:", embeddings.shape, "(chunks x numbers per chunk)")
print("first 8 numbers of chunk 0:", embeddings[0][:8].round(3))

In [ ]:
# Meaning, not matching words: "boat" vs "ship" share no letters, but are close in meaning.
pairs = [("a big boat on the sea", "a large ship in the ocean"),
         ("a big boat on the sea", "my favourite pizza topping")]
for a, b in pairs:
    va, vb = model.encode([a, b], normalize_embeddings=True)
    print(f"similarity = {float(va @ vb):.2f}   {a!r} vs {b!r}")

## 4. Vector store: save the vectors so we can search them

We use **ChromaDB** (in memory here). We ask for **cosine** distance, so `similarity = 1 - distance`.

In [ ]:
import chromadb, uuid

client = chromadb.Client()
try:
    client.delete_collection("workshop")
except Exception:
    pass
collection = client.create_collection("workshop", metadata={"hnsw:space": "cosine"})

collection.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=[{k: v for k, v in c.metadata.items() if isinstance(v, (str, int, float))} for c in chunks],
)
print("chunks stored:", collection.count())

## 5. Retrieval: find the most relevant chunks

Embed the **question** the same way, then ask the store for the nearest chunks.

In [ ]:
def retrieve(query, top_k=3, score_threshold=0.0):
    q = model.encode([query], normalize_embeddings=True)[0]
    res = collection.query(query_embeddings=[q.tolist()], n_results=top_k)
    hits = []
    for doc, meta, dist in zip(res["documents"][0], res["metadatas"][0], res["distances"][0]):
        score = 1 - dist
        if score >= score_threshold:
            hits.append({"score": round(score, 3), "source": meta.get("source_file"),
                         "page": meta.get("page"), "content": doc})
    return hits

question = "Which ship is nicknamed Old Ironsides?"   # <- change me!
pd.DataFrame(retrieve(question, top_k=3))

## 6. Generation: give the context to an LLM

The best chunks are the **context**. The **prompt** = instruction + context + question.
The instruction "say I don't know" stops the model from making things up.

In [ ]:
def build_prompt(context, question):
    return f"""Use the following context to answer the question concisely.
If the answer is not in the context, say "I don't know."

Context:
{context}

Question: {question}

Answer:"""

hits = retrieve(question, top_k=3)
context = "\n\n".join(h["content"] for h in hits)
prompt = build_prompt(context, question)
print(prompt)

In [ ]:
# The API key comes from (1) a Colab Secret named OPENAI_API_KEY, or (2) a hidden box below.
# It is never saved in the notebook.
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

if not os.getenv("OPENAI_API_KEY"):
    try:
        from google.colab import userdata          # Colab sidebar: key icon -> Secrets
        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    except Exception:
        os.environ["OPENAI_API_KEY"] = getpass("API key: ")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.1, max_tokens=512)
print(llm.invoke(prompt).content)

## 7. Put it together and experiment

Change the question, `top_k`, or the chunk size, and see whether the answer improves.
Then ask something the documents **cannot** answer and check for "I don't know".

In [ ]:
def ask(question, top_k=3):
    hits = retrieve(question, top_k=top_k)
    context = "\n\n".join(h["content"] for h in hits)
    answer = llm.invoke(build_prompt(context, question)).content
    print("Q:", question, "\nA:", answer, "\n\nSources:")
    for h in hits:
        print(f"  {h['score']:.2f}  {h['source']} (page {h['page']})")

ask("Which museum has a butterfly garden?")
ask("What is the attention mechanism in transformers?")   # not in the documents